# Prompting Techniques with Ollama

Prompting steers a pretrained LLM at inference time, with **no weight updates**. In this notebook you will:

1. See four techniques in action: **few-shot**, **chain-of-thought**, **self-consistency**, **RAG**
2. Practice them yourself in short exercises

| Technique | Idea |
|---|---|
| Zero/Few-shot | Give instructions, optionally with examples |
| Chain-of-Thought (CoT) | Ask for step-by-step reasoning before the answer |
| Self-Consistency | Sample several CoT chains, majority-vote the answer |
| RAG | Retrieve relevant documents and put them in the prompt |

## Setup
Run once (Colab with GPU runtime). It installs Ollama and pulls a small model.

In [58]:
%pip install -q ollama scikit-learn numpy
!sudo apt-get install -y -qq zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!nohup ollama serve > ollama.log 2>&1 &
import time; time.sleep(5)
!ollama pull gemma4:e2b

In [59]:
import ollama, re
from collections import Counter

MODEL = "gemma4:e2b"

def generate(prompt, max_tokens=150, temperature=0.0):
    """Send a prompt to the model and return its text. temperature=0 is deterministic."""
    r = ollama.generate(model=MODEL, prompt=prompt, think=False,
                        options={"temperature": temperature, "num_predict": max_tokens})
    return r["response"].strip()

print(generate("Say hello in one short sentence."))

Hello!


## Demo 1: Zero-shot vs. Few-shot
Few-shot adds input→output examples so the model copies the format.

In [60]:
review = "The pacing was dreadful, but the soundtrack was gorgeous."
task = "Classify the review as Positive, Negative, or Mixed. Reply with one word."

zero_shot = f"{task}\n\nReview: {review}\nLabel:"

few_shot = f"""{task}

Review: A masterpiece of modern cinema.
Label: Positive

Review: I fell asleep halfway through.
Label: Negative

Review: Great acting, but the plot made no sense.
Label: Mixed

Review: {review}
Label:"""

print("Zero-shot:", generate(zero_shot, max_tokens=10))
print("Few-shot :", generate(few_shot, max_tokens=10))

Zero-shot: Mixed
Few-shot : Mixed


## Demo 2: Chain-of-Thought
Asking the model to reason step by step helps on multi-step problems.

In [61]:
question = ("A store had 50 shirts. It sold 18 on Monday, got a shipment of 30, "
            "then sold half of what it had. How many shirts are left?")   # answer: 31

print("Direct:\n", generate(f"Q: {question}\nGive only the final number.\nA:", max_tokens=10))
print("\nZero-shot CoT:\n", generate(f"Q: {question}\nA: Let's think step by step.", max_tokens=200))

Direct:
 36

Zero-shot CoT:
 Here is the step-by-step solution:

**1. Starting Inventory:**
The store begins with **50** shirts.

**2. Sales on Monday:**
They sold 18 shirts.
$50 - 18 = 32$ shirts remaining.

**3. Shipment Received:**
They received a shipment of 30 shirts.
$32 + 30 = 62$ shirts total.

**4. Second Sale:**
They sold half of what they had (half of 62).
Half of 62 is $62 / 2 = 31$ shirts sold.

**5. Final Count:**
Subtract the second sale from the current total:
$62 - 31 = 31$ shirts left.

**Answer:** There are **31** shirts left.


## Demo 3: Self-Consistency
Sample several reasoning chains (temperature > 0) and take the most common final answer.

In [62]:
def final_number(text):
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else None

prompt = (f"Q: {question}\nThink step by step, then end with 'The answer is <number>.'\nA:")
answers = []
for _ in range(5):
    out = generate(prompt, max_tokens=250, temperature=0.8)
    answers.append(final_number(out))

print("Votes :", Counter(answers))
print("Winner:", Counter(answers).most_common(1)[0][0], "(truth: 31)")

Votes : Counter({'31': 5})
Winner: 31 (truth: 31)


## Demo 4: RAG
Retrieve the most relevant document (TF-IDF here; real systems use embeddings + a vector DB) and add it to the prompt.

In [63]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

docs = [
    "Acme Corp's refund policy: items can be returned within 45 days with a receipt.",
    "Acme Corp support hours are 9am-5pm ICT, Monday to Friday.",
    "Acme Corp's Pro plan costs $29 per month and includes 5 seats.",
    "Acme Corp headquarters are located in Phnom Penh, Cambodia.",
]
vec = TfidfVectorizer().fit(docs)
doc_vecs = vec.transform(docs)

def retrieve(query, k=1):
    scores = cosine_similarity(vec.transform([query]), doc_vecs).ravel()
    return [docs[i] for i in scores.argsort()[::-1][:k]]

def rag_answer(query, k=1):
    context = "\n".join(retrieve(query, k))
    return generate(f"Answer using only the context.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:")

q = "How many days do I have to return an item at Acme Corp?"
print("No context:", generate(f"Question: {q}\nAnswer:", max_tokens=60))
print("With RAG  :", rag_answer(q))

No context: I do not have access to specific return policies for Acme Corp. To find out how many days you have to return an item, you will need to check their official website, return policy page, or contact Acme Corp directly.
With RAG  : The context does not state how many days you have to return an item at Acme Corp.


---
# Hands-on Practice
Complete the `TODO`s. Re-run each cell and compare outputs. Small models are noisy, so try a few variations.

### Exercise 1: Few-shot extraction
Make the model turn a sentence into JSON with keys `name`, `city`, `age`. First try zero-shot, then add **2 examples** to get a clean, consistent format.

In [64]:
import json

text = "Sokha, a 29-year-old engineer, just moved to Siem Reap."

zero = f"""Extract name, city, and age as JSON.

Text: {text}
JSON:"""

prompt = f"""Extract name, city, and age from the text. Reply with raw JSON only, no markdown.

Text: Chetha, a 34-year-old nurse, lives in Kampong Cham.
JSON: {{"name": "Chetha", "city": "Kampong Cham", "age": 34}}

Text: Dora, a 22-year-old student, starts her first job in Phnom Penh.
JSON: {{"name": "Dora", "city": "Phnom Penh", "age": 22}}

Text: {text}
JSON:"""

print("Zero-shot:\n", generate(zero, max_tokens=60))
print("\nFew-shot:\n", generate(prompt, max_tokens=60))

out = generate(prompt, max_tokens=60)
print("\nParsed:", json.loads(out))

Zero-shot:
 ```json
{
  "name": "Sokha",
  "city": "Siem Reap",
  "age": 29
}
```

Few-shot:
 {"name": "Sokha", "city": "Siem Reap", "age": 29}

Parsed: {'name': 'Sokha', 'city': 'Siem Reap', 'age': 29}


### Exercise 2: Chain-of-Thought
The puzzle below often trips up direct answers. Compare a **direct** prompt with a **CoT** prompt. Which is right? (Answer: 0.05, i.e. 5 cents)

In [65]:
puzzle = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much is the ball?"

direct = f"Q: {puzzle}\nGive only the final answer, no explanation.\nA:"
cot    = (f"Q: {puzzle}\nLet's think step by step. Define the ball's price as x, "
          f"write the equation, solve it, and then state the final answer.\nA:")

print("Direct:", generate(direct, max_tokens=40))
print("CoT   :", generate(cot, max_tokens=450))

Direct: $0.05
CoT   : Here is the step-by-step solution:

**1. Define the variables:**
Let $x$ be the cost of the ball (in dollars).
Let $b$ be the cost of the bat (in dollars).

**2. Write the equations based on the problem statement:**
* **Equation 1 (Total cost):** The bat and the ball cost $1.10 in total.
   $$b + x = 1.10$$
* **Equation 2 (Cost difference):** The bat costs $1.00 more than the ball.
   $$b = x + 1.00$$

**3. Solve the system of equations:**
We can use the substitution method by substituting the expression for $b$ from Equation 2 into Equation 1:
$$(x + 1.00) + x = 1.10$$

**4. Simplify and solve for $x$ (the cost of the ball):**
Combine the $x$ terms:
$$2x + 1.00 = 1.10$$

Subtract $1.00$ from both sides:
$$2x = 1.10 - 1.00$$
$$2x = 0.10$$

Divide by 2:
$$x = \frac{0.10}{2}$$
$$x = 0.05$$

**5. State the final answer:**
The cost of the ball is $\$0.05$ (or 5 cents).

***
**Check (Optional):**
If the ball ($x$) costs $\$0.05$, then the bat ($b$) costs $\$0.05 + \$1.

In this case, both the direct prompt and the cot prompt give the correct answers of 0.05. Here, I change the max_tokens from 200 to 450 to get the full details of the step as max_tokens 200 ran out of token space.

### Exercise 3: Self-Consistency
Write a function `self_consistent(prompt, k)` that samples `k` answers at `temperature=0.8` and returns the majority answer. Test it on the puzzle above with `k=1` vs `k=7`. Does voting help?

*Hint:* ask the model to end with `Answer: <number>` so extracting the answer is easy.

In [68]:
import re
from decimal import Decimal, InvalidOperation
from collections import Counter

def self_consistent(prompt, k=5, verbose=False):
    """Sample k answers at T=0.8, extract each final number, return the majority answer."""
    if not isinstance(k, int) or isinstance(k, bool) or k < 1:
        raise ValueError("k must be a positive integer")
    answers = []
    for i in range(k):
        out = generate(prompt, max_tokens=500, temperature=0.8)
        m = re.findall(r"Answer\s*:[^\d+\-.]*([+-]?(?:\d+(?:\.\d+)?|\.\d+))\s*(cents?)?", out, flags=re.I)
        ans = None
        if m:
            num, unit = m[-1]
            try:
                val = Decimal(num) / (100 if unit else 1)      # "5 cents" -> 0.05
                ans = format(val.normalize(), "f")
            except InvalidOperation:
                pass
        if verbose:
            print(f"Sample {i+1}/{k}: {out}\n -> parsed: {ans}\n")
        answers.append(ans)

    counts = Counter(a for a in answers if a is not None)
    print("Votes:", dict(counts), "| unparsed:", answers.count(None))
    if not counts:
        print("No valid votes; inspect the responses (verbose=True).")
        return None
    top = max(counts.values())
    winners = [a for a in counts if counts[a] == top]          # first-seen order
    if len(winners) > 1:
        print("Tie between", winners, "-> returning the first seen")
    return winners[0]

cot_prompt = (f"Q: {puzzle}\n"
              "Let the ball cost x dollars, write the equation and solve it. "
              "Be concise (no LaTeX, under 80 words). "
              "End with exactly: Answer: <number in dollars>\nA:")

print("=== k=1 ===")
winner_1 = self_consistent(cot_prompt, k=1)
print("\n=== k=7 ===")
winner_7 = self_consistent(cot_prompt, k=7)


=== k=1 ===
Votes: {'0.05': 1} | unparsed: 0

=== k=7 ===
Votes: {'0.05': 7} | unparsed: 0


With k=1 the model answered 0.05 (correct). With k=7, all 7 sampled chains also gave 0.05, so the majority vote was 0.05. Voting did not improve accuracy here, because the model was already right on every chain; it only cost about 7× more compute. This puzzle is well known and easy for this model.

### Exercise 4: RAG
1. Add **two new facts** of your own to `docs` (re-fit the vectorizer after).
2. Ask a question that only your facts can answer.
3. Compare the answer with and without retrieval. What happens when the answer is **not** in the docs? Edit the RAG prompt so the model says *"I don't know"* in that case.

In [30]:
# TODO: extend docs, rebuild vec/doc_vecs, then test
# q = "..."
# print("No context:", generate(f"Question: {q}\nAnswer:", max_tokens=60))
# print("With RAG  :", rag_answer(q))

In [77]:
docs = docs + [
    "The Glimmerfin is a river fish found only in Lake Orvanta; it grows to 40 cm and glows blue at night.",
    "The Orvanta Lantern Festival is held on the third Saturday of September and lasts 5 days.",
]
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(docs)
doc_vecs = vec.transform(docs)

def rag_answer(query, k=2):
    context = "\n".join(retrieve(query, k))
    prompt = f"""Answer the question using ONLY the context below.
If the context does not contain the answer, reply exactly: I don't know.

Context:
{context}

Question: {query}
Answer in one sentence:"""
    return generate(prompt, max_tokens=60)

q = "Where can we find Glimmerfin, and how long can it grow to?"
print("Question  :", q)
print("No context:", generate(f"Question: {q}\nAnswer in one sentence:", max_tokens=60))
print("With RAG  :", rag_answer(q))

q_unknown = "How many visitors attend the Orvanta Lantern Festival?"
print("\nQuestion  :", q_unknown)
print("With RAG  :", rag_answer(q_unknown))

Question  : Where can we find Glimmerfin, and how long can it grow to?
No context: Please provide the text or context you are referring to so I can answer your question about Glimmerfin.
With RAG  : The Glimmerfin can be found only in Lake Orvanta, and it grows to 40 cm.

Question  : How many visitors attend the Orvanta Lantern Festival?
With RAG  : I don't know.


### Challenge (optional)
Pick a task from your own work (e.g. classify support tickets, summarize, translate to Khmer). Design a prompt, then improve it by adding **a role, a format constraint, and examples**. Note which change helped most.

---
## Recap
- **Few-shot**: examples fix format and style
- **CoT**: reasoning steps help multi-step problems
- **Self-consistency**: voting across samples improves reliability, at k× the cost
- **RAG**: grounds answers in your own data

Next: when prompting is not enough, **fine-tuning** updates the weights.